## MCORR AND CNMF
**Analysis of riboL1-GCaMP8m responses imaged with 16x objective at zoom 2x over 512x512 pixels, 15fps**

#### Define parameters


In [ ]:
# Paths

from pathlib import Path
data_path = [Path('D:/Data/2P/Scnn1aAi14_A2M0/12142023/TSeries-12142023-0944-003')]
export_path = Path('H:/Analysis/2P/Scnn1aAi14_A2M0/12142023/run3/mesmerize/')

# Motion correction parameters

params_mcorr = \
{
  'main':
    {
        'strides': [36, 36],
        'overlaps': [24, 24],
        'max_shifts': [16, 16],
        'max_deviation_rigid': 12,
        'border_nan': 'copy',
        'pw_rigid': True,
        'gSig_filt': None
    },
}

# CNMF parameters
params_cnmf =\
{
    'main': # indicates that these are the "main" params for the CNMF algo
        {
            'fr': 15, # framerate, very important!
            'p': 1,
            'nb': 2,
            'merge_thr': 0.8,
            'rf': 20,
            'stride': 12, # "stride" for cnmf, "strides" for mcorr
            'K': 8,
            'gSig': [6, 6],
            'ssub': 1,
            'tsub': 1,
            'method_init': 'greedy_roi',
            'min_SNR': 2.0,
            'SNR_lowest':  1.0,
            'rval_thr': 0.8,
            'rval_lowest': -1,
            'use_cnn': True,
            'min_cnn_thr': 0.9,
            'cnn_lowest': 0.1,
            'decay_time': 0.2,
        },
    'refit': True, # If `True`, run a second iteration of CNMF
}

# Extra parameters
params_extra = \
    {
        'cleanup': False # If `True`, run cleanup after CNMF, i.e., delete all df data
    }

# Concatenate the two dictionaries
params = {'params_mcorr': params_mcorr, 'params_cnmf': params_cnmf, 'params_extra': params_extra}
print(params)

#### Run MCORR and CNMF

In [ ]:
import pipeline_mcorr_cnmf as preproc
_ , batch_path = preproc.run_mcorr_cnmf(data_path, params, export_path)

### Load outputs

In [ ]:
# If cleanup was set to false in the params, you can load the batch file into Mesmerize:
# batch_path=Path('H:/Analysis/2P/Scnn1aAi14_A2M0/12132023/run3/mesmerize/batch_20240105-235525.pickle')
print(batch_path)
if params['params_extra']['cleanup'] == False:
    import mesmerize_core as mc
    df = mc.load_batch(batch_path)

### Visualize with fastplotlib


In [ ]:
import fastplotlib as fpl
cnmf_index = 1
rcm = df.iloc[cnmf_index].cnmf.get_rcm()
rcb = df.iloc[cnmf_index].cnmf.get_rcb()
residuals = df.iloc[cnmf_index].cnmf.get_residuals()
input_movie = df.iloc[cnmf_index].caiman.get_input_movie()

iw_rcm = fpl.ImageWidget(
    data=[input_movie, rcm, rcb, residuals], 
    grid_plot_kwargs={"size": (800, 600)}, 
    cmap="gnuplot2"
)
iw_rcm.show()

In [ ]:
iw_rcm.close()

### Visualize with `mesmerize-viz`

In [ ]:
print(f"Num accepted/rejected: {len(df.iloc[-1].cnmf.get_good_components())}, {len(df.iloc[-1].cnmf.get_bad_components())}")

In [ ]:
import mesmerize_viz
viz_simple = df.cnmf.viz(
    image_data_options=["max"],
)
viz_simple.show(sidecar=True)
# viz_simple.show()
viz_simple.image_widget.cmap = "gray"

In [ ]:
viz_simple.close()

#### Clean up export folder: stop/restart notebook, then run first and last cells

In [ ]:
import pipeline_mcorr_cnmf as preproc
batch_path=Path('H:/Analysis/2P/C57_O1M2/10022023/run7/mesmerize/batch_20231230-135733.pickle')
preproc.cleanup_files(batch_path, export_path)